# GELU: where does the time go?
## Lecture 004 coding quiz

Build the mental model first, then test it with code. Based on Thomas Viehmann's
[lecture notebook](cuda-mode-session-4.ipynb) and [slides](cuda-mode-2024-02-03.pdf), especially pages 16–19.

**Workflow:** predict → explain in plain English → implement → check → revise your explanation.
Replace `...` and `TODO` markers. An unfinished exercise intentionally raises an error;
work section by section rather than running the whole notebook immediately. No answer key is included.

Exercises 1–5 require only Python. Exercises 6–10 require PyTorch and an NVIDIA CUDA GPU;
exercise 7 additionally requires a CUDA toolkit with `nvcc`, a compatible C++ compiler, and Ninja,
as in the lecture environment. No GPU is provisioned by this notebook. Exercise 10 is a stretch exercise.
Benchmarks here are inference-only, not autograd implementations.

Keep these distinctions in mind: bytes versus operations, intensity versus throughput,
necessary versus avoidable traffic, and measured time versus an ideal hardware limit.


In [1]:
import math
import statistics
import time

# These are model assumptions from the lecture, not a query of your actual GPU.
LECTURE_BANDWIDTH_GB_S = 900.0  # decimal GB/s
FP32_BYTES = 4


## 1. Explain the code: one line, many jobs

Read the lecture's eager implementation; no execution is needed yet.

```python
def gelu_eager(x):
    return 0.5 * x * (1 + torch.tanh(
        math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)
    ))
```

Explain this to someone who knows Python but not GPUs:
1. Why does one Python expression not imply one GPU kernel?
2. Trace one intermediate tensor from its creation to its next use.
3. Which work would fusion remove, and which arithmetic would remain?
4. Why is “the GPU advertises trillions of operations per second” insufficient to predict runtime?

Do not assume every byte read by a kernel necessarily reaches DRAM: caches exist.


**Your explanation / prediction:**

1. because a single python expression may contain multiple arithmentic operations which create intermediate values that need to be written and then read by subsequent operations
2. this one i'm not sure about, but i'm assumeing 0.5*x would create a intermeidate tensor value that needs to be written and then read when it is multiplied by the total expressions in the paren
3. fusion would remove the writing and reading the of the intermediate bytes, all the of the arithmetic remains the same but the values are kept in register files instead of accessing memory
4. becuase our kernel design dictates the memory bandwitch usage which can prevent us from achieving this peak throughput

## 2. Fill the blanks: count traffic before counting FLOPs

Consider a deliberately simpler expression evaluated in three separate FP32 kernels:

```python
a = x * 2
b = a + 1
y = b * 3
```

Each kernel reads one tensor and writes one tensor. Scalars add no tensor-sized traffic.
Assume no cache reuse across kernels. Count **aggregate bytes**, not individual transactions.
Then compare with one kernel that keeps `a` and `b` in registers.


In [ ]:
n = 1024 * 1024
separate_passes = 3
fused_passes = 1
bytes_per_pass = n * 8  # one input read plus one output write, over all n values
separate_bytes = 24
fused_bytes = 0
traffic_reduction_factor = 

assert all(v is not Ellipsis for v in (
    separate_passes, fused_passes, bytes_per_pass,
    separate_bytes, fused_bytes, traffic_reduction_factor
)), "Fill every blank first."
assert separate_bytes == separate_passes * bytes_per_pass
assert fused_bytes == fused_passes * bytes_per_pass
assert fused_bytes == n * FP32_BYTES * 2
assert math.isclose(traffic_reduction_factor, separate_bytes / fused_bytes)
print(f"Separate: {separate_bytes / 1e6:.3f} MB; fused: {fused_bytes / 1e6:.3f} MB")


Why does reducing traffic by this factor **not guarantee** the same runtime speedup?


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


## 3. Implement the memory speed-of-light estimate

Implement a function returning microseconds from bytes and decimal GB/s.
Reject negative byte counts and nonpositive bandwidth. This is a memory-only ideal estimate:
assume the counted bytes reach main memory at peak bandwidth. It excludes launch overhead.


In [7]:

def memory_floor_us(byte_count, bandwidth_gb_s):
    # TODO: validate arguments and convert units correctly.
    if byte_count < 0 or bandwidth_gb_s <= 0: raise ValueError
    bytes_in_gb = byte_count/10**9
    time = bytes_in_gb / bandwidth_gb_s
    time_us = time * 10**6
    return time_us

assert math.isclose(memory_floor_us(1_000_000_000, 1), 1_000_000)
assert memory_floor_us(0, 900) == 0
for invalid in [(-1, 900), (1, 0), (1, -1)]:
    try:
        memory_floor_us(*invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError")

print("Fused GELU memory estimate (µs):",
      memory_floor_us(1024 * 1024 * 8, LECTURE_BANDWIDTH_GB_S))


Fused GELU memory estimate (µs): 9.320675555555557


If a repeated benchmark beats this estimate, explain why cache reuse or a wrong bandwidth assumption should be investigated before concluding that physics was violated.


**Your explanation / prediction:**

Because we are computing the theoretical minimum amount of time It would take to transfer a given number of bytes with a given bandwidth So if a benchmark beats this then some of our assumptions have to be incorrect because physics Is kind of a hard Rule and much less likely that we're violating physics laws

Note: we assumed all counted bytes travel through GPU main memory. If repeated runs retrieve inputs from a faster cache, dividing those bytes by main-memory bandwidth no longer gives the applicable limit.

## 4. Implement a two-resource roofline model

Assume perfect overlap between memory movement and compute. Return their individual times,
the larger time, and the limiting resource (`"memory"`, `"compute"`, or `"balanced"`).
Use decimal TFLOP/s. This toy model omits startup costs and assumes an applicable compute peak.
Do **not** treat `tanh` as one ordinary FLOP when applying this to real GELU.


In [12]:
def roofline_times(byte_count, flop_count, bandwidth_gb_s, compute_tflop_s):
    # TODO: calculate both times in microseconds.
    mem_us = memory_floor_us(byte_count, bandwidth_gb_s)
    flops_tb = flop_count / 10**12
    flop_us = (flops_tb / compute_tflop_s) * 10**6
    if math.isclose(mem_us, flop_us):
        bottleneck = "balanced"
    elif mem_us > flop_us:
        bottleneck = "memory"
    else:
        bottleneck = "compute"
    return {"flops_us": flop_us, "mem_us": mem_us, "ideal_us": max(flop_us, mem_us), "bottleneck": bottleneck}
    # Validate nonnegative work and positive rates.
    # Return a dict with keys: memory_us, compute_us, ideal_us, bottleneck.
    raise NotImplementedError("Implement roofline_times")

before = roofline_times(30_000, 12_000, 1, 0.001)
after = roofline_times(10_000, 12_000, 1, 0.001)
assert math.isclose(before["ideal_us"], 30)
assert math.isclose(after["ideal_us"], 12)
assert before["bottleneck"] == "memory"
assert after["bottleneck"] == "compute"
assert roofline_times(10_000, 10_000, 1, 0.001)["bottleneck"] == "balanced"
print("Speedup:", before["ideal_us"] / after["ideal_us"])


Speedup: 2.5


Explain why the speedup is less than the traffic reduction. Then explain why adding useless arithmetic can raise computational intensity without improving results per second.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


## 5. Explain the measurements: hypothesis versus proof

These are **synthetic observations**, not measurements from your GPU.

| Case | Observation |
|---|---|
| A | 1,000, 2,000, and 4,000 elements each take 5 µs |
| B | Doubling a large input doubles runtime |
| C | Useful bytes / runtime gives 80 GB/s on a nominal 900 GB/s device |
| D | Replacing GELU with a multiply changes runtime from 100 µs to 12 µs |

For each case, give (1) a plausible hypothesis, (2) something it does **not** prove,
and (3) a measurement or controlled experiment to do next.
Why does case D implicate the calculation without proving that advertised peak FLOP/s is saturated?


**Your explanation / prediction:**

A: 1. overhead dominates 2. does not prove compute or memory as constraint 3. increase size of inputs and rerun 
B: 1. Work proportional to input size dominates 2. either memory or computation could be limiting 3. use profiler for mem throughput and compute activity
C: 1.expensive computation limits throughput 2. no effective bandwith prove neither compute nor memory bound 3. replace gelu with simple mult, controling for reads, write and laungh config
D: 1. GELU’s calculation limits performance 2. The GPU’s advertised peak FLOP/s is saturated 3. Inspect instruction mix, compute activity, and register usage to understand why the calculation is costly. 

## GPU setup — run when ready for exercises 6–10

Use your existing lecture CUDA environment. Record its identity; do not assume it is the
lecture's RTX 3090. All following correctness checks use FP32 and no gradients.


In [13]:
import os
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

assert torch.cuda.is_available(), "Continue in the NVIDIA CUDA lecture environment."
device = torch.device("cuda", torch.cuda.current_device())
major, minor = torch.cuda.get_device_capability(device)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
# CC/CXX come from the lecture NGC Compose environment.
torch.manual_seed(4)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(device))

@torch.no_grad()
def gelu_eager(x):
    return 0.5 * x * (1 + torch.tanh(
        math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)
    ))


PyTorch: 2.4.0a0+3bcc3cddb5.nv24.07 CUDA: 12.5
GPU: NVIDIA L4


## 6. Fill the blanks: measure completed GPU work

Complete this CUDA-event timer. Events use **milliseconds**, but we want microseconds per call.
Warm up outside the timed interval, wait for the final event before reading its timestamp,
and report the median of repeated batches. Compilation must happen before calling this helper.

Predict: how would an unsynchronized CPU timer mislead us? Why can a CUDA-event interval
still contain GPU idle gaps caused by slow host submission? This helper is not a pure instruction-latency measurement.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


In [ ]:
def event_time_us(fn, warmup=10, repeats=100, batches=5):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(batches):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        # TODO: record start, run fn repeats times, record end, wait for end.
        raise NotImplementedError("Complete event ordering and unit conversion")
        us_per_call = ...
        samples.append(us_per_call)
    return statistics.median(samples)

x = torch.randn(1024 * 1024, device=device)
print("Eager µs:", event_time_us(lambda: gelu_eager(x)))
print("Built-in µs:", event_time_us(lambda: F.gelu(x, approximate="tanh")))


## 7. Implement the missing CUDA kernel body

Complete the TODO body inside the source string. Each thread should:
1. Compute its global element index and guard against going past `n`.
2. Read one FP32 input into a local value.
3. Evaluate the lecture's tanh approximation and write one output.

Use float literals (`0.5f`, etc.) and `tanhf`. Avoid allocating intermediate tensors.
The wrapper, input checks, device guard, and current-stream launch are supplied.
Explain why 257 elements is a useful correctness case when the block size is 256.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


In [ ]:
cuda_source = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cmath>

__global__ void quiz_gelu_kernel(const float* input, float* output, int64_t n) {
    // TODO: global index, bounds check, load, GELU calculation, store.
}

void quiz_gelu_out(torch::Tensor input, torch::Tensor output) {
    TORCH_CHECK(input.is_cuda() && output.is_cuda(), "CUDA tensors required");
    TORCH_CHECK(input.device() == output.device(), "Devices must match");
    TORCH_CHECK(input.scalar_type() == torch::kFloat32 &&
                output.scalar_type() == torch::kFloat32, "FP32 required");
    TORCH_CHECK(input.is_contiguous() && output.is_contiguous(), "Contiguous required");
    TORCH_CHECK(input.sizes() == output.sizes(), "Shapes must match");
    c10::cuda::CUDAGuard guard(input.device());
    int64_t n = input.numel();
    if (n == 0) return;
    constexpr int threads = 256;
    auto stream = at::cuda::getCurrentCUDAStream(input.get_device());
    quiz_gelu_kernel<<<(n + threads - 1) / threads, threads, 0, stream>>>(
        input.data_ptr<float>(), output.data_ptr<float>(), n);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
}
"""
assert "TODO" not in cuda_source, "Implement the kernel body and remove its TODO marker."
module = load_inline(
    name="lecture4_quiz_gelu",
    cpp_sources="void quiz_gelu_out(torch::Tensor input, torch::Tensor output);",
    cuda_sources=cuda_source,
    functions=["quiz_gelu_out"],
    verbose=False,
)


In [ ]:
# Correctness before performance. Never accept speed from a wrong or empty kernel.
for n in [0, 1, 31, 256, 257, 4099]:
    x = torch.linspace(-10, 10, n, device=device, dtype=torch.float32)
    out = torch.full_like(x, float("nan"))
    module.quiz_gelu_out(x, out)
    torch.testing.assert_close(out, F.gelu(x, approximate="tanh"),
                               atol=2e-6, rtol=2e-5)
print("Boundary and numerical checks passed.")


## 8. Implement a size sweep and interpret it

For each size, benchmark eager GELU, built-in tanh GELU, and your custom `out` kernel.
Allocate input/output before timing and check the custom result first.
The custom `out` API reuses output storage; the other APIs allocate outputs through PyTorch's
allocator. Label this API difference in your conclusions rather than attributing all gains to fusion.

Complete the loop below. Store element count, method, microseconds, and useful effective GB/s.
For the **custom kernel**, count `n * 8` useful bytes. Leave bandwidth as `None` for other methods:
the eager expression has intermediate traffic, so `n * 8 / time` would not measure its actual traffic.
Repeated buffers can hit cache; useful effective GB/s is not a DRAM counter.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


In [ ]:
rows = []
for n in [1024, 4096, 65536, 1048576, 4194304]:
    # TODO: allocate, check correctness, time the three methods using event_time_us,
    # and append dictionaries to rows with keys n, method, us, useful_gb_s.
    raise NotImplementedError("Implement the size sweep")

for row in rows:
    print(row)


Which sizes suggest fixed overhead? Does proportional growth prove a memory bottleneck?
Compare custom-kernel bandwidth with your **actual GPU's documented bandwidth**, citing your source.
If you have not looked it up, report useful bandwidth without a utilization percentage.
State one conclusion your measurements support and one they do not.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


## 9. Design and implement a controlled experiment

Make a second extension based on exercise 7, replacing only the GELU expression with `2.0f * x`.
Keep tensor sizes, FP32 dtype, launch geometry, stream handling, allocation strategy, and timer unchanged.
Use a different extension name. Check correctness against `2 * x` before benchmarking both kernels
on a large size from exercise 8. Retain the original GELU module for comparison.

Predict both outcomes: multiplication is much faster; multiplication takes about the same time.
Explain why neither outcome alone establishes a detailed hardware diagnosis.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


In [ ]:
# TODO: create the multiply extension, check correctness, and compare median timings.
raise NotImplementedError("Implement the controlled experiment")


## 10. Stretch: fuse GELU into its producer

The preceding operation is `z = 2 * x + 1`. Its only consumer is GELU.
Implement two custom kernels: a producer that writes `z`, and a combined producer+GELU kernel.
Use your existing GELU kernel for the separate path. Preallocate all outputs and intermediates.

Compare:
- Separate path: producer reads x/writes z; GELU reads z/writes y.
- Combined path: read x, calculate z locally, apply GELU, write y.

Complete the byte model, then implement and benchmark both paths with identical inputs.
Validate against `F.gelu(2 * x + 1, approximate="tanh")` before timing.


In [ ]:
def producer_gelu_bytes(n):
    separate = ...  # total useful bytes for BOTH kernels
    combined = ...
    saved = ...
    return separate, combined, saved

# TODO: implement extensions, validate, and benchmark separate versus combined paths.
raise NotImplementedError("Complete the byte model and producer fusion experiment")


**Teach it back:** Why can producer fusion improve the whole pipeline even if standalone GELU
already approaches its memory limit? What changes if another consumer needs the unmodified `z`?
Could fusion hurt by increasing register use? Separate the traffic prediction from your measured speedup.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._


## Wrap-up: your performance argument

In five sentences, explain:
1. What was unnecessary in the original eager execution?
2. What fusion changed about arithmetic, traffic, and launches.
3. Which hardware limit your measurements suggest, and the evidence.
4. Why high intensity or high utilization alone is not the goal.
5. What you would investigate next.

### References
- Thomas Viehmann, [CUDA-MODE lecture 004](https://github.com/gpu-mode/lectures/tree/main/lecture_004), accompanying local notebook and slides linked above.
- PyTorch [CUDA timing semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution).
- PyTorch [C++/CUDA extension tools](https://docs.pytorch.org/docs/stable/cpp_extension.html).

This quiz contains deliberately incomplete student code. Saved lecture timings are historical examples,
not expected answers or performance guarantees for your machine.


**Your explanation / prediction:**

_Write your answer here before running the next experiment._
